# Semana 9: RQ4, inferência no Rio Grande do Sul

Seção 1 de `prereg/plano_rq4.md` ("Inference") e as mudanças datadas no fim dele. Roda as 15 execuções do treino confirmatório (R1, R2 e R3, sementes 1 a 5, pesos em `resultados/semana5`) em 32 bits, com o código de montagem do modelo e de pré-processamento da semana 6, nas 575 janelas de 512 × 512 que entram em alguma das três distâncias da área mapeada da UFRGS (152 no 22JDM, 216 no 22JCN, 207 no 22JDN). Nada é ajustado no Rio Grande do Sul. **Este notebook não calcula nenhuma medida de concordância**: ele só grava as margens.

| O que roda | Detalhe |
|---|---|
| Entrada | As seis bandas do Sentinel-2 de 06/05/2024 na grade inteira de cada tile (conferidas pelo SHA-256 do registro), escala 1e-4 e normalização do módulo de dados do TerraTorch, como no treino |
| Pixels | Em cada janela, só os pixels atribuídos a ela (cada pixel do tile pertence à primeira janela, em ordem de linhas, que o cobre), na ordem: janelas por linha e coluna, pixels em ordem de linhas |
| Conferência do modelo | Antes do Rio Grande do Sul, cada execução roda na Bolívia e é comparada com as margens gravadas na semana 6 (diferença máxima abaixo de 0,05) |
| Saída | `resultados/semana9/rq4_margens`: `rotulos_<tile>.npz` (código, número de referências e estrato de cada pixel, nas três distâncias), `run_<regime>_s<semente>_<tile>.npz` (`margem` em float32 dos pixels pontuados em alguma distância; `margem_q` em int8, 4 × margem arredondada, de todos os pixels, só para mapas), `manifesto_sha256.csv`, `info_inferencia_rq4.json` |

A chave dos códigos já foi aberta na rodada confirmatória (`prereg/desvios.md`), e aqui os regimes aparecem pelo nome.

Abra o Colab logada na conta do mestrado, **com GPU T4**, e rode na ordem. Se a célula 1 pedir para reiniciar a sessão, reinicie e siga da célula 2. Se a sessão cair, ou se a RAM passar de 62%, a célula 7 para; reinicie e rode de novo da célula 2 em diante: as execuções já gravadas são puladas. Se alguma célula der erro, pare e guarde a saída inteira.

| Célula | O que faz |
|---|---|
| 1 | Instala TerraTorch 1.2.11 (mesmas versões da semana 6) |
| 2 | Monta o Drive, clona o repositório, confere a chave, os pesos e o espaço livre |
| 3 | Confere o backbone pelo hash |
| 4 | Baixa e confere os 15 chips da Bolívia e as margens da semana 6 para a conferência do modelo |
| 5 | Lê os tiles e as máscaras, confere tudo pelo SHA-256 e pela tabela de janelas, corta as janelas no disco local e grava os rótulos (10 a 20 min) |
| 6 | Definições do modelo (cópia da semana 6) |
| 7 | Inferência das 15 execuções (cerca de 1 hora) |
| 8 | Remonta o Drive, confere os arquivos e grava o manifesto |

Arquivos a conferir depois da célula 8, na pasta `resultados/semana9/rq4_margens`: `manifesto_sha256.csv` e `info_inferencia_rq4.json` (as margens ficam no Drive).

In [ ]:
# Célula 1 de 8: instalação
!pip install -q "terratorch[peft]==1.2.11" "torchgeo==0.7.1" "peft==0.21.2"
!pip uninstall -y -q torchao

from importlib.metadata import version
for pacote in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy'):
    print(f'{pacote:11s} {version(pacote)}')
assert version('terratorch') == '1.2.11', 'Versão do TerraTorch diferente da fixada.'

import sys
if 'numpy' in sys.modules and sys.modules['numpy'].__version__ != version('numpy'):
    print(f"\nREINICIE A SESSÃO: o NumPy carregado é o {sys.modules['numpy'].__version__} e o instalado é o {version('numpy')}.")
    print('Menu Ambiente de execução > Reiniciar sessão (não precisa excluir o ambiente) e rode a partir da célula 2.')
else:
    print('\nNumPy carregado e instalado coincidem. Pode seguir para a célula 2.')

In [ ]:
# Célula 2 de 8: Drive, repositório, chave e pesos
import os, re, json, time, hashlib, shutil, glob, gc, csv, subprocess, sys
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

S5 = f'{RAIZ}/resultados/semana5'
MARGENS6 = f'{RAIZ}/resultados/semana6/margens'
CHAVE6 = f'{RAIZ}/resultados/semana6_chave/chave_codigos.json'
HASHES_S1F11 = f'{RAIZ}/resultados/semana6/entradas_sen1floods11_sha256.csv'
S2DIR = f'{RAIZ}/dados/rs2024/s2'
MASCDIR = f'{RAIZ}/dados/rs2024/mascaras'
SAIDA = f'{RAIZ}/resultados/semana9/rq4_margens'      # pasta fixa: uma sessão nova continua de onde a anterior parou
LOCAL = '/content/rq4_inferencia'
for d in (SAIDA, LOCAL):
    os.makedirs(d, exist_ok=True)
for resto in glob.glob(f'{SAIDA}/parcial_*'):
    os.remove(resto)                                    # cópia interrompida de uma sessão anterior

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

def esperar(caminho, limite=180):
    t0 = time.time()
    while not os.path.exists(caminho):
        if time.time() - t0 > limite:
            raise FileNotFoundError(f'{caminho} não apareceu no Drive em {limite} s. Pare e guarde esta saída.')
        time.sleep(5)

def trazer(no_drive, sha_esperado, pasta=LOCAL):
    # Copia do Drive para o disco local e confere o SHA-256; se não conferir, remonta o Drive uma vez e copia de novo.
    local = f'{pasta}/{os.path.basename(no_drive)}'
    for tentativa in (1, 2):
        esperar(no_drive)
        if not (os.path.exists(local) and sha256_arquivo(local) == sha_esperado):
            shutil.copyfile(no_drive, local + '.parcial')
            os.replace(local + '.parcial', local)
        sha = sha256_arquivo(local)
        if sha == sha_esperado:
            return local
        if tentativa == 1:
            print(f'   {os.path.basename(no_drive)}: SHA-256 diferente do registro; remontando o Drive e copiando de novo.')
            drive.flush_and_unmount()
            time.sleep(10)
            drive.mount('/content/drive', force_remount=True)
    raise ValueError(f'{no_drive}: SHA-256 {sha[:16]}..., o registro diz {sha_esperado[:16]}... Pare e guarde esta saída.')

def gravar_no_drive(local, nome):
    # Copia um arquivo pronto do disco local para a SAIDA com nome provisório e só então dá o nome final.
    parcial = f'{SAIDA}/parcial_{nome}'
    shutil.copyfile(local, parcial)
    os.replace(parcial, f'{SAIDA}/{nome}')

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO_GIT}/analysis/rq4')
import s2_exportacao as S
import mascaras as M
import inferencia as I
SHA_MODULOS = {m: sha256_arquivo(f'{REPO_GIT}/analysis/rq4/{m}') for m in ('mascaras.py', 'inferencia.py')}

# Chave dos códigos da semana 6, aberta na rodada confirmatória (prereg/desvios.md)
CHAVE = json.load(open(CHAVE6))
assert CHAVE == {'8tjg': 'R1', '4m9f': 'R2', '6wke': 'R3'}, 'A chave da semana 6 não é a registrada em prereg/desvios.md.'
CODIGO = {r: c for c, r in CHAVE.items()}

REGIMES = ('R1', 'R2', 'R3')
SEMENTES = (1, 2, 3, 4, 5)
PESOS = {}
for regime in REGIMES:
    for semente in SEMENTES:
        boas = []
        for tentativa in (1, 2):
            registro = f'{S5}/exec_{regime}_s{semente}_t{tentativa}.json'
            if os.path.exists(registro) and json.load(open(registro)).get('situacao') == 'ok':
                boas.append(registro)
        pesos = f'{S5}/pesos_{regime}_s{semente}.pt'
        assert len(boas) == 1 and os.path.exists(pesos), f'Faltam arquivos do treino {regime} s{semente} em {S5}.'
        PESOS[(regime, semente)] = dict(pesos=pesos, bytes=os.path.getsize(pesos))
print(f'{len(PESOS)} execuções de treino encontradas.')

# Registros do repositório
REG_S2 = json.load(open(f'{REPO_GIT}/analysis/rq4/entradas/rq4_entradas.json'))
SHA_S2 = {r['arquivo']: r['sha256'] for r in csv.DictReader(open(f'{REPO_GIT}/analysis/rq4/entradas/rq4_entradas_sha256.csv'))}
REG_MASC = {r['arquivo']: r for r in csv.DictReader(open(f'{REPO_GIT}/analysis/rq4/mascaras/rq4_mascaras_sha256.csv'))}
TAB = list(csv.DictReader(open(f'{REPO_GIT}/analysis/rq4/mascaras/rq4_janelas.csv')))
GRADES = {e['tile']: e for e in REG_S2['entradas']}
assert list(GRADES) == list(S.TILES)
JANELAS = {t: I.janelas_da_inferencia(TAB, t) for t in S.TILES}
assert [len(JANELAS[t]) for t in S.TILES] == [152, 216, 207], {t: len(v) for t, v in JANELAS.items()}
for t in S.TILES:
    em2k = {r['window_id'] for r in TAB if r['tile'] == t and r['distancia_m'] == '2000' and r['kept'] == 'True'}
    assert {w for w, _, _ in JANELAS[t]} == em2k, f'{t}: as janelas que entram em alguma distância não são as de 2 km.'

livre = shutil.disk_usage(MEUDRIVE)[2]
print(f'Espaço livre no Drive: {livre / 1e9:.0f} GB. As margens ocupam cerca de 7 GB.')
assert livre > 10e9, 'Espaço livre no Drive insuficiente para as margens (mínimo de 10 GB).'
print(f'Repositório no commit {commit[:7]}. Janelas: {sum(len(v) for v in JANELAS.values())}. Saída em {SAIDA.replace(MEUDRIVE, "MyDrive")}')

In [ ]:
# Célula 3 de 8: backbone
from huggingface_hub import hf_hub_download

REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
ARQUIVO = 'Prithvi_EO_V2_300M_TL.pt'
REVISAO = '63adbd39c271da4c42f447e69b1a7c91a338cdc9'
SHA_BACKBONE = '3629cedfbb350faafcb0dac902ae0d3c927e25ce8d9e0024aa1276ec66956ddb'

no_drive = f'{RAIZ}/modelo_fixado/backbone/{ARQUIVO}'
if not os.path.exists(no_drive):
    print('Backbone ausente no Drive; baixando na revisão fixada.')
    hf_hub_download(REPO, ARQUIVO, revision=REVISAO, local_dir=os.path.dirname(no_drive))
BACKBONE_PT = f'/content/{ARQUIVO}'
if not (os.path.exists(BACKBONE_PT) and sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE):
    shutil.copy(no_drive, BACKBONE_PT)
assert sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE, 'sha256 do backbone diferente do registrado em modelo.md.'
print(f'Backbone conferido ({SHA_BACKBONE[:8]}…), cópia local em {BACKBONE_PT}')

In [ ]:
# Célula 4 de 8: Bolívia para a conferência do modelo
import urllib.request, concurrent.futures as cf
import rasterio

BUCKET = 'https://storage.googleapis.com/sen1floods11/v1.1/'
DADOS = '/content/sen1floods11'
S2 = f'{DADOS}/v1.1/data/flood_events/HandLabeled/S2Hand'
LB = f'{DADOS}/v1.1/data/flood_events/HandLabeled/LabelHand'
for d in (S2, LB):
    os.makedirs(d, exist_ok=True)
TAM = 512
IDX_BANDAS = [1, 2, 3, 8, 11, 12]          # B2, B3, B4, B8A, B11, B12 nos 13 canais do Sen1Floods11
ESCALA = np.float32(0.0001)

hashes = {l['arquivo']: l['sha256'] for l in csv.DictReader(open(HASHES_S1F11, newline=''))}
LAB_BOL = np.load(f'{MARGENS6}/labels_bolivia.npz', allow_pickle=False)
CHAVES_BOL = [str(c) for c in LAB_BOL['chaves']]
assert len(CHAVES_BOL) == 15

def baixar(trabalho):
    url, destino = trabalho
    nome = os.path.basename(destino)
    if not (os.path.exists(destino) and sha256_arquivo(destino) == hashes[nome]):
        for tentativa in range(4):
            try:
                urllib.request.urlretrieve(url, destino + '.parcial')
                os.replace(destino + '.parcial', destino)
                break
            except Exception:
                time.sleep(2 * (tentativa + 1))
    assert sha256_arquivo(destino) == hashes[nome], f'{nome} diferente do hash gravado na semana 6.'
    return nome

trabalhos = []
for chave in CHAVES_BOL:
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/S2Hand/{chave}_S2Hand.tif', f'{S2}/{chave}_S2Hand.tif'))
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/LabelHand/{chave}_LabelHand.tif', f'{LB}/{chave}_LabelHand.tif'))
with cf.ThreadPoolExecutor(16) as pool:
    list(pool.map(baixar, trabalhos))

def preencher(x, valor):
    x = np.ma.filled(x.astype(np.float32), np.nan)
    return np.where(np.isnan(x), np.float32(valor), x).astype(np.float32)

def ler_chip(chave):
    with rasterio.open(f'{S2}/{chave}_S2Hand.tif') as s:
        assert s.count == 13 and (s.height, s.width) == (TAM, TAM), f'{chave}: imagem'
        x = preencher(s.read(masked=True), 0.0)[IDX_BANDAS]
    with rasterio.open(f'{LB}/{chave}_LabelHand.tif') as s:
        y = preencher(s.read(1, masked=True), -1.0).astype(np.int16)
    return x, y

BOL_X = np.empty((15, len(IDX_BANDAS), TAM, TAM), np.float32)
BOL_Y = np.empty((15, TAM, TAM), np.int16)
for i, c in enumerate(CHAVES_BOL):
    BOL_X[i], BOL_Y[i] = ler_chip(c)
pont = BOL_Y >= 0
assert np.array_equal(pont.reshape(15, -1).sum(1), LAB_BOL['tamanhos']) and np.array_equal(BOL_Y[pont].astype(np.uint8), LAB_BOL['rotulo']), (
    'Rótulos da Bolívia diferentes dos da semana 6.')

# Margens da semana 6 na Bolívia, conferidas pelo manifesto
manifesto6 = {l['arquivo']: l['sha256'] for l in csv.DictReader(open(f'{MARGENS6}/manifesto_sha256.csv', newline=''))}
BOL6 = {}
for regime in REGIMES:
    for semente in SEMENTES:
        nome = f'run_{CODIGO[regime]}_s{semente}_bolivia.npz'
        local = trazer(f'{MARGENS6}/{nome}', manifesto6[nome])
        z = np.load(local, allow_pickle=False)
        assert [str(c) for c in z['chaves']] == CHAVES_BOL and np.array_equal(z['tamanhos'], LAB_BOL['tamanhos'])
        BOL6[(regime, semente)] = z['margem'].copy()
print(f'Bolívia: 15 chips conferidos pelos hashes da semana 6; margens das 15 execuções da semana 6 conferidas pelo manifesto.')

In [ ]:
# Célula 5 de 8: tiles, máscaras e janelas
from rasterio.windows import Window

def carregar_npz(caminho):
    with np.load(caminho, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

def ler_mascara(tile, nome):
    r = REG_MASC[nome]
    local = trazer(f'{MASCDIR}/{nome}', r['sha256'])
    with rasterio.open(local) as s:
        assert (s.height, s.width) == (S.LADO, S.LADO) and str(s.crs) == 'EPSG:32722', nome
        assert tuple(s.transform)[:6] == tuple(GRADES[tile]['grade']['transform']), nome
        dados = s.read()
    assert M.sha_conteudo(dados) == r['sha256_conteudo'], f'{nome}: valores diferentes do registro.'
    os.remove(local)
    return dados

NOVOS, JAN_X, PONTUADO, TAMANHOS, INFO_TILES = {}, {}, {}, {}, {}
t_inicio = time.time()
for tile in S.TILES:
    t0 = time.time()
    jan = JANELAS[tile]
    novos = I.atribuidos(S.LADO, S.LADO, jan)
    cod, nrefs = {}, {}
    for d in M.DISTANCIAS_M:
        a = ler_mascara(tile, f'{tile}_classe_{d}m.tif')
        cod[d], nrefs[d] = a[0], a[1]
    estr = ler_mascara(tile, f'{tile}_estrato.tif')[0]
    rot = I.rotulos_tile(cod, nrefs, estr, jan, novos)
    n_conf = I.conferir_com_tabela(rot, TAB, tile)
    del cod, nrefs, estr

    # Janelas do Sentinel-2 no disco local (uint16, 6 bandas), com a fração sem dado conferida pela tabela
    e = GRADES[tile]
    rel = f'canonico/{tile}_{e["id"]}_s2.tif'
    s2_local = trazer(f'{S2DIR}/{rel}', SHA_S2[rel])
    tab2k = {r['window_id']: r for r in TAB if r['tile'] == tile and r['distancia_m'] == '2000'}
    X = np.lib.format.open_memmap(f'{LOCAL}/janelas_{tile}.npy', mode='w+', dtype=np.uint16, shape=(len(jan), 6, TAM, TAM))
    with rasterio.open(s2_local) as s:
        assert (s.count, s.height, s.width) == (6, S.LADO, S.LADO) and tuple(s.transform)[:6] == tuple(e['grade']['transform'])
        for i, (w, lin, col) in enumerate(jan):
            X[i] = s.read(window=Window(col, lin, TAM, TAM))
            frac = float(np.all(X[i] == 0, axis=0).mean())
            assert abs(frac - float(tab2k[w]['frac_missing_input'])) < 1e-12, f'{w}: fração sem dado diferente da tabela'
    X.flush()
    os.remove(s2_local)

    # Rótulos do tile: gravados uma vez; numa sessão nova, conferidos contra os do Drive
    nome = f'rotulos_{tile}.npz'
    if os.path.exists(f'{SAIDA}/{nome}'):
        antigo = carregar_npz(trazer(f'{SAIDA}/{nome}', sha256_arquivo(f'{SAIDA}/{nome}')))
        assert set(antigo) == set(rot) and all(np.array_equal(antigo[k], rot[k]) for k in rot), (
            f'{nome} do Drive é diferente do recalculado. Pare e guarde esta saída.')
        situacao = 'já existia, igual ao recalculado'
    else:
        np.savez_compressed(f'{LOCAL}/{nome}', **rot)
        gravar_no_drive(f'{LOCAL}/{nome}', nome)
        situacao = 'gravado'
    NOVOS[tile], JAN_X[tile] = novos, np.load(f'{LOCAL}/janelas_{tile}.npy', mmap_mode='r')
    PONTUADO[tile], TAMANHOS[tile] = rot['pontuado'], rot['tamanhos']
    INFO_TILES[tile] = {'janelas': len(jan), 'pixels_atribuidos': int(rot['tamanhos'].sum()),
                        'pixels_pontuados_em_alguma_distancia': int(rot['pontuado'].sum()),
                        'pares_janela_distancia_conferidos': n_conf}
    print(f'{tile}: {len(jan)} janelas, {INFO_TILES[tile]["pixels_atribuidos"]:,} pixels atribuídos, '
          f'{INFO_TILES[tile]["pixels_pontuados_em_alguma_distancia"]:,} pontuados em alguma distância; '
          f'{n_conf} contagens conferidas com rq4_janelas.csv; {nome} {situacao} ({time.time() - t0:.0f} s)')
    del rot
    gc.collect()
print(f'Entradas prontas em {(time.time() - t_inicio) / 60:.0f} min. Nenhum modelo foi rodado ainda.')

In [ ]:
# Célula 6 de 8: modelo (cópia da semana 6)
import warnings, contextlib, io, logging, gc
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from importlib.metadata import version
from scipy.special import log_expit, logsumexp
try:
    from terratorch.tasks import SemanticSegmentationTask
    from terratorch.datamodules import Sen1Floods11NonGeoDataModule
except ModuleNotFoundError as erro:
    raise RuntimeError('TerraTorch não está instalado nesta sessão. Rode a célula 1 e confira que ela imprime terratorch 1.2.11.') from erro

warnings.filterwarnings('ignore', category=UserWarning)
assert torch.cuda.is_available(), 'Sem GPU: Ambiente de execução > Alterar tipo de ambiente de execução > T4.'
DISPOSITIVO = 'cuda'
BANDAS = ['BLUE', 'GREEN', 'RED', 'NIR_NARROW', 'SWIR_1', 'SWIR_2']
ESPERADO_M = {'R1': 15.1, 'R2': 20.6, 'R3': 319.0}
TOLERANCIA_M = 0.1
LR = {'R1': 5e-4, 'R2': 5e-4, 'R3': 5e-5}
EPOCAS = 50
LOTE_INF = 8
LIMITE_DIF_VAL = 0.05

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

def datamodule():
    para_tensor = lambda: ToTensorV2(transpose_mask=False)
    return Sen1Floods11NonGeoDataModule(
        data_root=DADOS, batch_size=LOTE_INF, num_workers=0, bands=BANDAS,
        train_transform=[para_tensor()], val_transform=[para_tensor()], test_transform=[para_tensor()],
        drop_last=False, constant_scale=0.0001, no_data_replace=0.0, no_label_replace=-1, use_metadata=False)

LORA = dict(method='LORA', replace_qkv='qkv',
            peft_config_kwargs=dict(target_modules=['qkv.q_linear', 'qkv.v_linear', 'mlp.fc1', 'mlp.fc2'],
                                    lora_alpha=16, r=16))

def tarefa(regime):
    args = dict(
        backbone='prithvi_eo_v2_300_tl', backbone_pretrained=True, backbone_ckpt_path=BACKBONE_PT,
        backbone_bands=BANDAS,
        necks=[{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]}, {'name': 'ReshapeTokensToImage'}],
        decoder='UperNetDecoder', decoder_channels=256, decoder_scale_modules=True,
        head_dropout=0.1, num_classes=2, rescale=True)
    if regime == 'R2':
        args['peft_config'] = LORA
    t = SemanticSegmentationTask(
        model_factory='EncoderDecoderFactory', model_args=args, loss='ce', ignore_index=-1,
        lr=LR[regime], optimizer='AdamW', optimizer_hparams={'weight_decay': 0.05},
        scheduler='CosineAnnealingLR', scheduler_hparams={'T_max': EPOCAS, 'eta_min': 0},
        freeze_backbone=(regime == 'R1'), plot_on_val=0)
    t.configure_models()
    treinaveis = sum(p.numel() for p in t.parameters() if p.requires_grad)
    if regime == 'R2':
        n_lora = sum(1 for n, _ in t.model.encoder.named_parameters() if 'lora_A' in n)
        assert n_lora == 96, 'Número de módulos LoRA diferente do esperado.'
    assert abs(treinaveis / 1e6 - ESPERADO_M[regime]) <= TOLERANCIA_M, 'Parâmetros treináveis fora do esperado.'
    return t

DM = datamodule()

@contextlib.contextmanager
def em_silencio():
    """Esconde o que a construção do modelo imprime, que diferiria entre os regimes."""
    logging.disable(logging.CRITICAL)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)

def carregar_modelo(regime, semente):
    with em_silencio():
        t = tarefa(regime)
        t.model.to(DISPOSITIVO)
        gc.collect()
        pesos = torch.load(PESOS[(regime, semente)]['pesos'], map_location=DISPOSITIVO)
        do_modelo = set(t.model.state_dict().keys())
        esperado = do_modelo if regime != 'R1' else {k for k in do_modelo if not k.startswith('encoder.')}
        assert set(pesos.keys()) == esperado, 'Os tensores salvos não coincidem com os do modelo.'
        resultado = t.model.load_state_dict(pesos, strict=False)
        assert not resultado.unexpected_keys, 'Tensores salvos sem lugar no modelo.'
        assert set(resultado.missing_keys) == (do_modelo - esperado), 'Tensores do modelo sem peso.'
        modelo = t.model.to(DISPOSITIVO).float().eval()
        for p in modelo.parameters():
            p.requires_grad_(False)
        del t, pesos
    gc.collect()
    return modelo

def margens_lote(modelo, x):
    # Margem z_água − z_terra de um lote (B, 6, 512, 512) de números digitais, em 32 bits, como na semana 6.
    with torch.no_grad(), torch.autocast(device_type=DISPOSITIVO, enabled=False):
        xt = torch.from_numpy(np.asarray(x).astype(np.float32) * ESCALA).to(DISPOSITIVO)
        y = torch.zeros((xt.shape[0], TAM, TAM), dtype=torch.int64, device=DISPOSITIVO)
        xt = DM.aug({'image': xt, 'mask': y})['image']
        assert xt.dtype == torch.float32 and xt.ndim == 4 and xt.shape[1] == len(BANDAS), tuple(xt.shape)
        z = modelo(xt).output.float()
        assert tuple(z.shape[-2:]) == (TAM, TAM), f'Saída {tuple(z.shape)}'
        return (z[:, 1] - z[:, 0]).cpu().numpy()

def conferir_bolivia(modelo, regime, semente):
    partes = []
    for i in range(0, 15, LOTE_INF):
        m = margens_lote(modelo, BOL_X[i:i + LOTE_INF])
        for j in range(m.shape[0]):
            partes.append(m[j][BOL_Y[i + j] >= 0].astype(np.float32))
    nova = np.concatenate(partes)
    antiga = BOL6[(regime, semente)]
    assert nova.shape == antiga.shape
    return float(np.max(np.abs(nova - antiga))), int(np.sum((nova > 0) != (antiga > 0)))

def margens_tile(modelo, tile):
    X, novos = JAN_X[tile], NOVOS[tile]
    partes = []
    for i in range(0, len(novos), LOTE_INF):
        m = margens_lote(modelo, X[i:i + LOTE_INF])
        for j in range(m.shape[0]):
            partes.append(m[j][novos[i + j]].astype(np.float32))
    return np.concatenate(partes)

print('Definições carregadas. Nada foi inferido ainda.')

In [ ]:
# Célula 7 de 8: inferência das 15 execuções
import psutil
LIMITE_RAM_PCT = 62

def arquivo_run(regime, semente, tile):
    return f'run_{regime}_s{semente}_{tile}.npz'

def feito(regime, semente, tile):
    nome = arquivo_run(regime, semente, tile)
    return os.path.exists(f'{SAIDA}/{nome}') and os.path.exists(f'{SAIDA}/{nome}.sha256')

for regime in REGIMES:
    for semente in SEMENTES:
        faltam = [t for t in S.TILES if not feito(regime, semente, t)]
        if not faltam:
            print(f'{regime} s{semente}: já feita.')
            continue
        pct = psutil.virtual_memory().percent
        if pct > LIMITE_RAM_PCT:
            raise RuntimeError(f'PAREI: RAM em {pct:.0f}%. Reinicie a sessão e rode da célula 2 em diante; o que está feito é pulado.')
        t0 = time.time()
        modelo = carregar_modelo(regime, semente)
        dif, trocas = conferir_bolivia(modelo, regime, semente)
        assert dif <= LIMITE_DIF_VAL, f'{regime} s{semente}: diferença de {dif:.3f} na Bolívia contra a semana 6. Pare e guarde esta saída.'
        json.dump({'regime': regime, 'semente': semente, 'diferenca_maxima': dif, 'pixels_com_sinal_trocado': trocas,
                   'gpu': torch.cuda.get_device_name(0)},
                  open(f'{SAIDA}/conferencia_bolivia_{regime}_s{semente}.json', 'w'), indent=2)
        for tile in faltam:
            todas = margens_tile(modelo, tile)
            assert todas.dtype == np.float32 and todas.size == int(TAMANHOS[tile].sum()) and np.isfinite(todas).all(), (
                f'{regime} s{semente} {tile}: margens com tamanho errado ou não finitas; nada foi gravado.')
            nome = arquivo_run(regime, semente, tile)
            if os.path.exists(f'{SAIDA}/{nome}'):          # gravação interrompida numa sessão anterior: o arquivo fica de lado
                os.replace(f'{SAIDA}/{nome}', f'{SAIDA}/descartado_{nome}')
            np.savez_compressed(f'{LOCAL}/{nome}', margem=todas[PONTUADO[tile]], margem_q=I.quantizar(todas))
            sha = sha256_arquivo(f'{LOCAL}/{nome}')
            gravar_no_drive(f'{LOCAL}/{nome}', nome)
            with open(f'{SAIDA}/{nome}.sha256', 'w') as f:
                f.write(sha + '\n')
            os.remove(f'{LOCAL}/{nome}')
            del todas
        print(f'{regime} s{semente}: Bolívia com diferença máxima {dif:.1e} ({trocas} pixels com sinal trocado); '
              f'{len(faltam)} tile(s) gravado(s) em {time.time() - t0:.0f} s (RAM em {psutil.virtual_memory().percent:.0f}%).')
        del modelo
        gc.collect(); torch.cuda.empty_cache()
print('\nGuarde a saída desta célula.')

In [ ]:
# Célula 8 de 8: conferência final
from importlib.metadata import version

print('Gravando o Drive e conferindo de novo; pode levar alguns minutos.')
drive.flush_and_unmount()
time.sleep(10)
drive.mount('/content/drive', force_remount=True)

linhas, ruins = [], []
for tile in S.TILES:
    nome = f'rotulos_{tile}.npz'
    esperar(f'{SAIDA}/{nome}')
    linhas.append([nome, os.path.getsize(f'{SAIDA}/{nome}'), sha256_arquivo(f'{SAIDA}/{nome}')])
    for regime in REGIMES:
        for semente in SEMENTES:
            nome = f'run_{regime}_s{semente}_{tile}.npz'
            assert os.path.exists(f'{SAIDA}/{nome}') and os.path.exists(f'{SAIDA}/{nome}.sha256'), (
                f'Falta {nome}. Rode a célula 7 de novo.')
            sha = sha256_arquivo(f'{SAIDA}/{nome}')
            if sha != open(f'{SAIDA}/{nome}.sha256').read().strip():
                ruins.append(nome)
                continue
            linhas.append([nome, os.path.getsize(f'{SAIDA}/{nome}'), sha])
for nome in ruins:
    os.replace(f'{SAIDA}/{nome}', f'{SAIDA}/descartado_{nome}')
    os.replace(f'{SAIDA}/{nome}.sha256', f'{SAIDA}/descartado_{nome}.sha256')
assert not ruins, (f'{len(ruins)} arquivo(s) no Drive diferente(s) do que foi gravado: {ruins}. Eles foram renomeados para '
                   f'descartado_...; rode a célula 7 de novo (só eles serão refeitos) e depois esta.')

with open(f'{SAIDA}/manifesto_sha256.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['arquivo', 'bytes', 'sha256'])
    w.writerows(linhas)
BOL = [json.load(open(f'{SAIDA}/conferencia_bolivia_{r}_s{s}.json')) for r in REGIMES for s in SEMENTES]
INFO = {'plano': 'prereg/plano_rq4.md, secao 1 e mudancas datadas', 'commit_repositorio': commit, 'modulos_sha256': SHA_MODULOS,
        'regimes': list(REGIMES), 'sementes': list(SEMENTES), 'codigos_semana6': CODIGO,
        'pesos': {f'{r}_s{s}': {'arquivo': os.path.basename(v['pesos']), 'bytes': v['bytes']} for (r, s), v in PESOS.items()},
        'backbone_sha256': SHA_BACKBONE, 'precisao_inferencia': 'float32', 'lote': LOTE_INF,
        'tiles': INFO_TILES, 'conferencia_bolivia': BOL,
        'margem_q': 'int8, round(4 m) limitado a -127..127, de todos os pixels atribuidos; so para mapas',
        'versoes': {q: version(q) for q in ('terratorch', 'torchgeo', 'peft', 'torch', 'numpy')},
        'gpu': torch.cuda.get_device_name(0), 'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA}/info_inferencia_rq4.json', 'w'), indent=2, ensure_ascii=False)
print(f'{len(linhas)} arquivos conferidos depois de remontar o Drive (3 de rótulos e 45 de margens).')
print(f'Diferença máxima na Bolívia entre as 15 execuções: {max(b["diferenca_maxima"] for b in BOL):.1e}.')
print('Manifesto e info_inferencia_rq4.json gravados em resultados/semana9/rq4_margens. Nenhuma medida de concordância foi calculada.')